# 03 - Black-box Optimisation
---

### **Black-box Problems**
Some optimisation problems involve an objective function which can be evaluated but cannot easily be expressed, differentiated or optimised analytically. Such objective functions are often refered to as 'black-boxes'; we know what we put into the box and what we get out but we do not properly understand what goes on 'in the box' to turn the inputs into the output. Black-box problems often have objectives which are noisy, computationally expensive, non-smooth or simulation-based. Additionally, it is often not possible to obtain a guarantee of global optimality. 

There are several techniques discussed below to optimise black-box problems. The setup for each of them is roughly the same; namely we have the following:

- An objective function $f(x_1, ... , x_k)$ we seek to maximise - for minimise, we maximise $-f(x_1, ... , x_k)$)
- Input variables $x_1, ... , x_n$ we seek the optimal values for - these could be integers or floats

### **Random Search**
The idea behind random search is to test many combinations of input variables which are generated by sampling randomely.It works as follows:

1. For each variable $x_i$ define a distribution to draw from. Often this is a uniform distribution to make all values in a range equally likely. 
2. For each variable $x_i$ draw $n$ random samples from the probability distributions deifined in step 1
3. Evaluate the objective function $f(x)$ for each of the $n$ sets of input values
4. Choose the input variables which maximise the objective

The benefits of random search are that it is simple and easy to implement and, since each set of input variables is evaluated independently, the process can be parallelised for increased speed. However, this independence is a double-edged sword because the process does not learn from its previous evaluations so they do not serve to improve the optimisation algorithm. Additionally, if the search space is large, many iterations will be required to have confidence in the optimality of the solution. 


### **Grid Search**
One of the issues with random search is that the randomely sampled values may not cover the entire search space. Grid search seeks to rectify this problem by rather than sampling randomely, generate a lattice of candidate points accross the search space which are all evaluated. It works as follows:

1. For each variable $x_i$ choose a range to draw from and a step size. Use this to derivate candidate points at equal intervals. For example, if the range was $[0, 10]$ and the step size was $2$ then the candidate points would be $\{0, 2, 4, 6, 8, 10\}$. 
2. Generate all combinations of candidate points for all variables $x_1, ... , x_k$
3. Evaluate the objective function $f(x)$ for each of the sets of input values
4. Choose the input values which maximise the objective

As discussed, grid search provides greater coverage across the search space, improving confidence in the optimality of the solution. However, choosing the step-size involves a careful trade-off; too large and you risk overshooting the optimal value, too small and you must evaluate many combinations which could be slow. Note that as with random search, the evaluations are independent so parallelism could improve the solve time. 

### **Coarse-to-Fine**
The coarse-to-fine optimisation algorithm seeks to fix the issue around choosing the step size in grid search. The basic idea is to start with a grid search with a large step size and then iteratively decrease the step size, zooming in on the most promising region. It works as follows:

1. First perform a grid search with a large step size
2. Tighten the range by reducing the step size such that the centre of the range in the optimal value found in step 1
3. Repeat the process, finding the optimal value with the reduced step size before tightening the range and reducing the step size further.
4. Stop once a minimum step size is reached

Coarse-to-fine does go a long way to addressing the issues around step size in grid search although if the initial step size is too large it may still not find the optimal value. Nevertheless, coarse-to-fine will usually explore the search space more efficiently so is generally preferable to grid search. 

### **Bayesian Optimisation**
In the case of random and grid search the points are evaluated completely independently; we take no learnings from previously evaluated inputs. In coarse-to-fine we use the previously evaluated points to narrow the search range. However, in all cases we do not attempt to estimate the uncertainty around how we expect candidate points to perform. This is particularly problematic in cases where each evaluation of the objective function is expensive, so we want to learn as much as possible from every point we test. 

Rather than evaluating many candidate points blindly, Bayesian optimisation builds a probabilistic surrogate model of the objective function, often using a Gaussian process. A surrogate model is a cheap-to-evaluate approximation of the true objective function. We use it because the real objective may require running an expensive simulation, training a machine learning model, or carrying out a physical experiment. Instead of repeatedly evaluating the expensive objective across the whole search space, we fit the surrogate model to the points we have already observed and use it to estimate what the objective might look like between those observed points. 

Because the surrogate model is probabilistic, it gives more than a single predicted objective value. For each candidate point it can provide both a prediction of the objective and an estimate of the uncertainty around that prediction. Typically we have high confidence close to points we have already evaluated and lower confidence in unexplored regions. This makes the surrogate model useful not only for predicting where good solutions may lie, but also for identifying where we still need more information. 

The acquisition function then uses the surrogate model to decide which point to evaluate next. It is a scoring rule that balances exploitation and exploration. Exploitation means sampling points where the surrogate model predicts a high objective value, while exploration means sampling points where the model is uncertain and there may still be hidden improvements. The point with the best acquisition score is then chosen for the next expensive evaluation. It works as follows:

1. Begin by evaluating the objective function $f(x)$ at a small number of initial points (often generated randomly)
2. Fit a surrogate model to the observed input-output pairs so that the model gives both a predicted objective value and an estimate of uncertainty
3. Use an acquisition function to score candidate points based on the surrogate model, balancing exploration and exploitation
4. Evaluate the true objective function at the point with the best acquisition score
5. Update the surrogate model with the new observation and repeat until a stopping criterion is reached

The key advantage of Bayesian optimisation is that it is usually far more sample-efficient than random search or grid-based approaches, making it well suited to expensive simulations and hyperparameter tuning. However, it is more complicated to implement, each new evaluation depends on the previous ones so parallelism is less straightforward, and the surrogate model can become computationally expensive or inaccurate in very high-dimensional problems. 

### **Genetic Algorithm**
A genetic algorithm implements an approach with has many similarities with survival of the fittest as observed in nature. The best candidate points 'survive' each generation and are combined together to create new candidate points. It works as follows:

1. Generate an initial population of candidate inputs at random and evaluate them using the objective
2. Select the stronger candidates to act as parents for the next generation. Better-performing candidates are more likely to be selected, although some weaker candidates may still be retained to preserve diversity.
3. Create a new generation of candidate points by combining information from pairs of parent solutions. This crossover step mixes parts of good solutions in the hope that their useful features can be combined into even better offspring.
4. Apply random mutations to some of the new candidate points by changing one or more input values slightly. Mutation helps the algorithm continue exploring the search space and reduces the risk of getting stuck around a poor local optimum.
5. Repeat the evaluate-select-crossover-mutate cycle for many generations, or until a stopping criterion is met such as a maximum number of generations or little further improvement.
7. Choose the best candidate solution found across all generations

The advantage of genetic algorithms is that they can work well on complex, non-smooth search spaces and do not require derivatives. They can also be effective when the search space contains many local optima or mixed variable types. However, they often require many objective evaluations, their performance depends heavily on choices such as population size and mutation rate, and they do not guarantee that the global optimum will be found. 